# Contextual DDM with formulas

This example regresses DDM drift rate (`v`) and non-decision time (`tau`) on trial-level covariates. `Formula` turns the expressions into vectorized NumPy operations before the DDM is simulated.

In [12]:
import matplotlib.pyplot as plt
import numpy as np

import superstats as sup
from superstats.transition import RandomWalk

## Simulate trial context

The context simulator returns one covariate sequence per simulated dataset. In an application, this function can instead draw from an experimental design or resample observed trial metadata.

In [13]:
def simulate_context(*, batch_size, num_steps):
    cue_validity = np.broadcast_to(np.linspace(-1.0, 1.0, num_steps), (batch_size, num_steps))
    n_cues = np.broadcast_to(1.0 + (np.arange(num_steps) % 3), (batch_size, num_steps))
    return {"cue_validity": cue_validity, "n_cues": n_cues}

context = sup.ContextSimulator(simulate_context)

## Specify regressions and the DDM

In [14]:
formula = sup.Formula([
    "v = v_0 + b_v * cue_validity",
    "tau = tau_0 + b_tau * n_cues",
])

prior = sup.JointPrior(
    v_0=sup.Prior("normal", loc=0.0, scale=2),
    b_v=RandomWalk(bounds=(-4.0, 4.0), initial_prior=sup.Prior("normal", loc=0.0, scale=0.5), sigma=sup.Prior("halfnormal", loc=0.0, scale=0.1)),
    tau_0=RandomWalk(bounds=(0.1, 0.8), initial_prior=sup.Prior("normal", loc=0.35, scale=0.03), sigma=sup.Prior("halfnormal", loc=0.0, scale=0.1)),
    b_tau=sup.Prior("normal", loc=0.0, scale=0.5),
    a=1.5,
    bias=0.5,
)

model = sup.Model(
    prior=prior,
    simulator=sup.simulation.sample_ddm,
    missing=None,
    context=context,
    context_mapping=sup.ContextMapping(formula_context=("cue_validity", "n_cues")),
    formula=formula,
)

## Sample and inspect

The sampled output retains latent baseline parameters and context. The resolved DDM parameters below are reconstructed with the same formulas for visualization.

In [16]:
sample = model.sample(batch_size=3, num_steps=75)

sample.keys()

dict_keys(['response_time', 'choice', 'time_steps', 'cue_validity', 'n_cues', 'b_v', 'tau_0', 'b_v_sigma', 'tau_0_sigma', 'v_0', 'b_tau'])